# Report Q4 by channel group

*Data Analysis in Python · Topic 12.4*

Your team wants Q4 2025's sessions, transactions and revenue by channel group, straight from GA4. Pull the report with GA4's client library, and turn it into a table of numbers.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the cell that installs `google-analytics-data`, the parameters cell and the cell that creates `client`. The client asks Simmer's practice endpoint, which answers like GA4 with Simmer Store's data.
2. In the first cell marked ✏️, create `response`: a report for `START_DATE` to `END_DATE`, with the dimension `sessionDefaultChannelGroup` and the metrics `sessions`, `transactions` and `purchaseRevenue`, in this order.
3. In the second cell marked ✏️, create the `channels` table: one row per channel group, with the API names as column names. Convert `sessions` and `transactions` to whole numbers and `purchaseRevenue` to decimal numbers.
4. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `channels` table has 9 channel groups, with 486,702 sessions, 14,997 transactions and $1,302,854.69 of revenue. Organic Search has the most sessions, 114,514.

In [ ]:
%pip install google-analytics-data

## Parameters

In [ ]:
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get("SIMMER_API_KEY")  # Colab Secrets, as in topic 12.1
GA4_ENDPOINT = "https://api.teamsimmer.com"  # Simmer's practice endpoint, not Google's
PROPERTY_ID = "123456789"  # Any number works on the practice endpoint. Use your own property's ID at work.
START_DATE, END_DATE = "2025-10-01", "2025-12-31"

In [ ]:
# The GA4 client, pointed at Simmer's practice endpoint. Run it, and leave it as it is.
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import DateRange, Dimension, Metric, RunReportRequest

client = BetaAnalyticsDataClient(transport="rest", client_options={"api_endpoint": GA4_ENDPOINT, "api_key": API_KEY})

## The report

In [ ]:
# ✏️ Create response


## The table

In [ ]:
# ✏️ Create channels


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source and not source.lstrip().startswith("%pip")] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    response = found.get("response")
    if response is None or not hasattr(response, "rows"):
        results.append((False, "response should be the answer of client.run_report(request)."))
    else:
        headers = [h.name for h in response.dimension_headers] + [h.name for h in response.metric_headers]
        if headers != ["sessionDefaultChannelGroup", "sessions", "transactions", "purchaseRevenue"]:
            results.append((False, f"The report should have the dimension sessionDefaultChannelGroup and the metrics sessions, transactions and purchaseRevenue, in this order. It has {headers}."))
        elif response.row_count != 9:
            results.append((False, f"The report should have 9 rows, one per channel group. It has {response.row_count}."))
        else:
            results.append((True, "A report of 9 channel groups, with sessions, transactions and purchaseRevenue."))

    channels = found.get("channels")
    columns = ["sessionDefaultChannelGroup", "sessions", "transactions", "purchaseRevenue"]
    if not isinstance(channels, pd.DataFrame) or [column for column in columns if column not in channels.columns]:
        results.append((False, f"channels should be a DataFrame with the columns {', '.join(columns)}."))
    elif len(channels) != 9:
        results.append((False, f"The channels table should have 9 rows, one per channel group. It has {len(channels)}."))
    elif not all(pd.api.types.is_numeric_dtype(channels[column]) for column in columns[1:]):
        results.append((False, "The metrics are still text: GA4 sends every value as a string. Convert sessions and transactions with astype(int) and purchaseRevenue with astype(float)."))
    elif int(channels["sessions"].sum()) == 486702 and abs(channels["purchaseRevenue"].sum() - 1302854.69) > 0.01:
        results.append((False, f"purchaseRevenue should add up to $1,302,854.69. It adds up to ${channels['purchaseRevenue'].sum():,.2f}. Check that the metrics are in the order you asked for them."))
    elif int(channels["sessions"].sum()) != 486702 or int(channels["transactions"].sum()) != 14997:
        results.append((False, f"Q4 2025 has 486,702 sessions and 14,997 transactions. The channels table has {channels['sessions'].sum():,} and {channels['transactions'].sum():,}. Check START_DATE and END_DATE in the request."))
    else:
        results.append((True, "486,702 sessions, 14,997 transactions and $1,302,854.69 of revenue in Q4 2025, the same as the Simmer API's report for the quarter."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
request = RunReportRequest(
    property=f"properties/{PROPERTY_ID}",
    dimensions=[Dimension(name="sessionDefaultChannelGroup")],
    metrics=[Metric(name="sessions"), Metric(name="transactions"), Metric(name="purchaseRevenue")],
    date_ranges=[DateRange(start_date=START_DATE, end_date=END_DATE)],
)
response = client.run_report(request)
response.row_count
```

```python
names = [h.name for h in response.dimension_headers] + [h.name for h in response.metric_headers]
rows = []
for row in response.rows:
    values = [v.value for v in row.dimension_values] + [v.value for v in row.metric_values]
    rows.append(values)
channels = pd.DataFrame(rows, columns=names)
channels[["sessions", "transactions"]] = channels[["sessions", "transactions"]].astype(int)
channels["purchaseRevenue"] = channels["purchaseRevenue"].astype(float)
channels
```

</details>